# Dataset Exploration

This notebook performs an initial inspection of the dice detection dataset.

Goals:
- verify project paths
- inspect Pascal VOC annotations
- visualize annotated images
- analyze class distribution and object counts

In [ ]:
from pathlib import Path
from utils.voc import parse_voc_xml
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from project_config import (
    PROJECT_ROOT,
    IMAGES_DIR,
    ANNOTATIONS_DIR,
)

In [ ]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("ANNOTATIONS_DIR exists:", ANNOTATIONS_DIR.exists())

image_paths = sorted(IMAGES_DIR.glob("*.jpg"))
xml_paths = sorted(ANNOTATIONS_DIR.glob("*.xml"))

print("Images:", len(image_paths))
print("Annotations:", len(xml_paths))
print("Sample image:", image_paths[0] if image_paths else "none")
print("Sample xml:", xml_paths[0] if xml_paths else "none")

In [ ]:
def show_image_with_boxes(image_path: Path, annotation: dict, figsize=(8, 10)):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image_rgb)

    for obj in annotation["objects"]:
        xmin, ymin, xmax, ymax = obj["bbox"]
        class_name = obj["class_name"]

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(rect)

        ax.text(
            xmin,
            max(0, ymin - 10),
            class_name,
            color="white",
            fontsize=10,
            bbox=dict(facecolor="red", alpha=0.7, pad=2),
        )

    ax.set_title(f'{annotation["filename"]} | objects: {len(annotation["objects"])}')
    ax.axis("off")
    plt.show()

In [ ]:
if not xml_paths:
    raise ValueError(f"No XML annotation files found in: {ANNOTATIONS_DIR}")

sample_xml = xml_paths[0]
sample_ann = parse_voc_xml(sample_xml)
sample_image_path = IMAGES_DIR / sample_ann["filename"]

show_image_with_boxes(sample_image_path, sample_ann)

In [ ]:
class_counter = Counter()
objects_per_image = []

for xml_path in xml_paths:
    ann = parse_voc_xml(xml_path)
    objects_per_image.append(len(ann["objects"]))

    for obj in ann["objects"]:
        class_counter[obj["class_name"]] += 1

print("Class distribution:", class_counter)
print("Average objects per image:", sum(objects_per_image) / len(objects_per_image))
print("Min objects:", min(objects_per_image))
print("Max objects:", max(objects_per_image))

In [ ]:
classes = sorted(class_counter.keys(), key=lambda x: int(x))
counts = [class_counter[c] for c in classes]

plt.figure(figsize=(8, 5))
plt.bar(classes, counts)
plt.title("Class distribution")
plt.xlabel("Dice value")
plt.ylabel("Count")
plt.grid(axis="y")
plt.show()